# Repository data and reproducible backtest

This is the main data walkthrough for QuantHaxs. Run every cell from a fresh Python kernel to catalog the local files, see reported extraction and OCR gaps, and reproduce the bundled backtest twice.

The bundled backtest uses **synthetic engineering data**, SHA-256-pinned inputs, and the existing options strategy engine. It needs no API key, network download, GPU, OCR, or model training. Historical real-data backtests retain their original qualification and availability limits.

[Data and reproduction guide](../docs/repository-data-workflow.md) Â· [Live Massive challenge notebook](../gator-quant-hacks-8k-options-challenge.ipynb) Â· [Source contract](../docs/team-data/source-contract.md)

Large and licensed source data stays on disk. This notebook provides its catalog and selected explicitly identified tables; it does not embed the entire corpus.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import version
import hashlib
import json
import sys
import uuid

candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in candidates if (p / "src/main.py").is_file()
             and (p / "examples").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook inside the QuantHaxs repository.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from src.repository_data_catalog import inventory_data_files, data_area_summary, audit_exported_run
from src.backtest_replay import reproduce

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:8]
REPORT_DIR = ROOT / "data/processed/repository_notebook" / RUN_ID
REPORT_DIR.mkdir(parents=True, exist_ok=False)
print("Repository found: QuantHaxs")
print("Fresh report directory:", REPORT_DIR.relative_to(ROOT).as_posix())
print("Mode: local metadata catalog + synthetic engineering replay")

## 1. Environment and storage organization

The receipt below records the actual interpreter and direct package versions. For the tested setup, use Python 3.11 and requirements-notebook-lock.txt. OCR and GPU dependencies belong to their separate workloads.

| Area | Meaning |
| --- | --- |
| data/raw | Original documents, quotes and acquisition records |
| .massive_cache | Original Massive API responses when available |
| data/processed | Extracted text, evidence, derived datasets and run receipts |
| artifacts | Retained study reports and diagnostic outputs |
| configs | Study parameters and registered-source metadata |
| examples | Small documentation examples and synthetic fixtures |
| references | Challenge briefs and starter reference material |
| Root metadata files | Taxonomies, tickers and discovery classifications |

Source bytes remain in their existing locations. File counts measure storage coverage, not extracted documents, observations, usable market rows, or accepted features.

Read the inventory as a storage map. A present file can be unreviewed, incomplete or unsuitable for historical use. A missing file supplies no signal or outcome. The source registration table below describes discovery/extraction status, not source coverage or accepted training labels.

In [ ]:
runtime = {"python": sys.version.split()[0], "executable": sys.executable,
           "packages": {name: version(name) for name in
                        ("numpy", "pandas", "requests", "matplotlib",
                         "nbformat", "nbclient", "ipykernel", "jupyterlab")}}
display(pd.DataFrame(runtime["packages"].items(), columns=["package", "installed_version"]))
(REPORT_DIR / "environment.json").write_text(json.dumps(runtime, indent=2) + "\n", encoding="utf-8")

In [ ]:
file_records = inventory_data_files(ROOT)
catalog = pd.DataFrame(file_records, columns=["area", "path", "extension", "bytes", "status", "note"])
areas = pd.DataFrame(data_area_summary(ROOT, file_records))
catalog.to_csv(REPORT_DIR / "data_inventory.csv", index=False)
areas.to_csv(REPORT_DIR / "data_areas.csv", index=False)
display(areas)
print("Cataloged files:", len(catalog))
print("Full catalog:", (REPORT_DIR / "data_inventory.csv").relative_to(ROOT).as_posix())
if not catalog.empty:
    display(catalog.groupby(["area", "extension"], dropna=False).size().rename("files").reset_index())
    display(catalog.groupby("area", sort=True).head(3))

The full inventory is exported as data_inventory.csv, including every file in the defined data areas. Use the cell below to browse a particular area. Its metadata scan skips symlinks and reads no raw or protected data contents.

In [ ]:
AREA_TO_BROWSE = "raw"  # raw, processed, artifacts, examples, references, configs, api_cache, supporting
display(catalog.loc[catalog["area"] == AREA_TO_BROWSE].head(25))

## 2. Data dictionary and source lineage

| Object | Grain and required identity | Meaning |
| --- | --- | --- |
| Batch manifest | Document; document_id, local source_path, known source SHA-256 | A URL is metadata; processing needs local original bytes |
| Transcript | Document; source/text hashes, normalization/version, method, engine/adapter/settings | Extraction quality and completeness remain explicit |
| Evidence | Exact transcript and character span; quoted_text equals the normalized slice | Retains auditable source attribution |
| Events | Issuer/accession/ticker/event date; pre/post sessions | Filing date alone is not an acceptance timestamp |
| Selected option legs | Event Ã— bucket Ã— leg; contract/strike/expiry/multiplier/selection date | Frozen selections consumed by the replay |
| Option bars | Contract Ã— observed session; close and volume | Last-trade marks; executable bid/ask and market capacity are separate |
| Results | Event Ã— bucket Ã— entry Ã— OTM Ã— horizon | Gross unit P&L using the existing implementation |
| Capacity | Event Ã— bucket Ã— strategy | Independent event-level cost, loss, capital and volume assumptions |
| Manifest | One run; settings, environment, input/code/output hashes | Evidence of reproduction; financial acceptance is separate |

Preserve raw/text hashes, original identities and definitions. Publication, receipt, download and processing times are distinct. Unknown timestamps, quantities, OCR confidence, model predictions and downstream acceptance remain unknown.

[Team data guide](../docs/team-data/README.md) Â· [Transcript contract](../src/document_transcript.py) Â· [Evidence contract](../src/document_evidence.py)

In [ ]:
registrations = []
source_root = ROOT / "configs/sources"
if source_root.is_dir():
    for metadata_path in sorted(source_root.glob("*/source.json")):
        if metadata_path.is_symlink():
            continue
        try:
            metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
            registrations.append({"source_id": metadata.get("source_id", metadata_path.parent.name),
                                  "stage": metadata.get("stage", "unknown"),
                                  "engine": metadata.get("engine", "unknown"),
                                  "registration": metadata_path.relative_to(ROOT).as_posix()})
        except (OSError, ValueError) as error:
            registrations.append({"source_id": metadata_path.parent.name, "stage": "unreadable",
                                  "engine": None, "registration": str(error)})
display(pd.DataFrame(registrations, columns=["source_id", "stage", "engine", "registration"]))
if not registrations:
    print("No source registrations are present in this checkout; this is not a zero-document corpus.")
print("A discovery registration or successful extraction does not establish accepted benchmark features.")

## 3. OCR and incomplete coverage

**Team-reported project limitation:** OCR and model processing over the full PDF dataset could not be completed with the GPU compute and processing resources available to the team. The extracted and reviewed information is incomplete, so the entire dataset and planned analysis could not be examined.

This portable checkout does not include private machine readiness reports or independently measured full-corpus coverage. Coverage, OCR accuracy, training completion and financial acceptance remain unknown unless established by the relevant source records. Missing documents are not empty observations, neutral sentiment, zero cash flows or zero returns. This notebook starts no OCR or model jobs.

Continued OCR data is unavailable because processing stopped short of full PDF coverage. Access/runtime and resource readiness, human gold-label review, exact historical document versions/clocks and downstream acceptance still need evidence. These are distinct gaps; this notebook does not diagnose one hardware failure as their sole cause. Native HTML with usable text can bypass OCR, but it does not fill missing scanned tables or guarantee complete extraction. Model coverage and historical review remain incomplete. The retained subset is not a representative full-corpus sample; missing text must never become a neutral signal or zero financial value.

In [ ]:
coverage = pd.DataFrame([
    {"measure": "Full PDF collection processed", "status": "incomplete (team reported)"},
    {"measure": "Extracted information reviewed", "status": "partial (team reported)"},
    {"measure": "Full-corpus OCR accuracy", "status": "unknown"},
    {"measure": "Trained prediction model completion", "status": "unknown"},
    {"measure": "Financial qualification", "status": "not established by this demo"},
])
display(coverage)
coverage.to_csv(REPORT_DIR / "ocr_coverage_status.csv", index=False)
print("No private audit records are required for the offline demonstration.")

## 4. Historical runs: evidence versus available data

Any local Massive smoke manifests describe historical runs. Their CSV row counts remain historical claims until local bytes pass their manifest hash check. A clean checkout may contain none of these runtime artifacts. Missing tables, unsealed manifests and LFS pointer stubs remain visible.

Post Benchmark retains its canonical account and qualification workflow. Exploratory reports, consumer diagnostics or zero eligible events cannot be converted into qualified portfolio performance here.

In [ ]:
historical_audits = []
for name in ("smoke", "smoke-options-native"):
    for row in audit_exported_run(ROOT / "data/processed" / name):
        historical_audits.append({"run": name, **row})
historical = pd.DataFrame(historical_audits)
display(historical)
historical.to_csv(REPORT_DIR / "historical_table_audit.csv", index=False)
print("Expected rows come from historical manifests; verified/missing status refers to current local bytes.")

## 5. Frozen offline backtest

The bundled DEMO snapshot is synthetic. Its selected events, option legs, bars and replay settings are hashed before evaluation. The adapter calls the existing evaluate, scoreboard, capacity and export functions. It refuses changed inputs, incompatible frozen settings, and an existing output directory.

The timing rule uses the first trading session strictly after a filing date for post entry. Pre entry is a hypothetical comparison. The existing synthetic-spot, fixed-horizon, stale-mark and carry assumptions remain visible in the frozen settings and receipts.

To reproduce from the repository root:

    python scripts/reproduce_backtest.py --snapshot-dir examples/backtest_replay --output-dir data/processed/replay-check/run-1
    python scripts/reproduce_backtest.py --snapshot-dir examples/backtest_replay --output-dir data/processed/replay-check/run-2 --compare data/processed/replay-check/run-1

Choose new output directory names for another attempt.

In [ ]:
SNAPSHOT = ROOT / "examples/backtest_replay"
snapshot_files = []
for path in sorted(SNAPSHOT.glob("*")):
    if path.is_file() and path.suffix in (".json", ".csv"):
        raw = path.read_bytes()
        snapshot_files.append({"input": path.name, "bytes": len(raw),
                               "sha256": hashlib.sha256(raw).hexdigest()})
display(pd.DataFrame(snapshot_files))
print("Snapshot provenance: synthetic engineering data; no original PDFs or market observations.")

### Asset, period and regime context

The next table derives dates and asset IDs from the bundled files, before evaluating any outcome. DEMO and FAKE-* are fictional assets. The dates locate the simulation on a calendar; they provide no measured bull/bear, volatility, liquidity or interest-rate regime for those assets. No real AMT trade is represented here. We do not classify regimes using later prices or select the event because its result looked favorable.

The options date-only event has no accepted publication timestamp: post entry is the first session strictly after its filing date; pre entry is hypothetical. Frozen leg selection, stale-price carry and synthetic spot limit interpretation. The wording workflow uses UTC-supported public bounds plus processing latency; entry and exit follow the actual reference-session schedule. A configured 252-session valuation grid is not 252 independent event observations.

In [ ]:
option_events = pd.read_csv(SNAPSHOT / "events.csv", dtype={"ticker": str})
option_legs = pd.read_csv(SNAPSHOT / "option_legs.csv")
option_bars = pd.read_csv(SNAPSHOT / "option_bars.csv")
option_settings = json.loads((SNAPSHOT / "replay.json").read_text())["settings"]
wording_workflow = json.loads((ROOT / "configs/simple_wording_workflow-v1.json").read_text())
wording_fixture = json.loads((ROOT / wording_workflow["mode_policy"]["engineering"]["fixture"]["path"]).read_text())
wording_protocol = json.loads((ROOT / wording_workflow["protocol"]["path"]).read_text())
wording_calendar = json.loads((ROOT / wording_workflow["calendar"]["path"]).read_text())
context_rows = []
for _, event in option_events.iterrows():
    context_rows.append({"workflow": "options", "asset": event["ticker"],
        "event_or_public_clock": event["event_date"], "entry_rule": f"pre hypothetical {event['t_pre']}; post {event['t_0']}",
        "observed_window": f"{option_bars['session'].min()} to {option_bars['session'].max()}",
        "horizon": str(option_settings["horizons"]) + " sessions; expiry " + str(option_legs['expiration_date'].max()),
        "status": "fictional synthetic; date-only clock", "regime": "not measured"})
for candidate in wording_fixture["candidates"]:
    quotes = [q for q in wording_fixture["quotes"] if q["instrument_id"] == candidate["security_id"]]
    context_rows.append({"workflow": "wording", "asset": candidate["security_id"],
        "event_or_public_clock": candidate["public_by_utc"], "entry_rule": wording_protocol["decision_schedule"],
        "observed_window": f"{min(q['at_utc'] for q in quotes)} to {max(q['at_utc'] for q in quotes)}",
        "horizon": wording_protocol["holding_rule"], "status": "FABRICATED engineering", "regime": "not measured"})
asset_period_context = pd.DataFrame(context_rows)
display(asset_period_context)
asset_period_context.to_csv(REPORT_DIR / "asset_period_context.csv", index=False)
print("Wording valuation calendar:", wording_calendar["date_from"], "to", wording_calendar["date_to"])
print("Sample: one invented options event; three fabricated wording candidates; no representative asset regime.")


In [ ]:
first_dir = REPORT_DIR / "replay-1"
second_dir = REPORT_DIR / "replay-2"
first_receipt = reproduce(SNAPSHOT, first_dir)
second_receipt = reproduce(SNAPSHOT, second_dir)
first_hashes = {name: info["sha256"] for name, info in first_receipt["exported_tables"].items()}
second_hashes = {name: info["sha256"] for name, info in second_receipt["exported_tables"].items()}
if first_hashes != second_hashes:
    raise AssertionError("Independent replay scientific-table hashes differ.")
display(pd.DataFrame([
    {"table": name, "rows": info["rows"], "sha256": info["sha256"],
     "independent_replay": "identical"} for name, info in first_receipt["exported_tables"].items()
]))
display(pd.DataFrame(audit_exported_run(first_dir)))
print("Independent replay verified: every scientific CSV has the same SHA-256.")
print("Mode:", first_receipt.get("status", "synthetic engineering replay"))
print("Economic qualification:", first_receipt.get("economic_qualification", "not_established"))

## 6. Inspect the mechanics and assumptions

The result table is gross P&L per dollar of synthetic stock at entry. It does not deduct portfolio trading costs or establish net investor returns. The capacity table separately models per-event capital, risk, volume participation and premium haircuts. Simultaneous positions do not have an account allocation/NAV model in this demonstration.

The sample contains one invented event. Its scoreboard is descriptive; no sample-size, Sharpe, inference, performance ranking, OCR-derived signal or trading recommendation is supported.

Read `entry`, `horizon` and expiry bucket before comparing a result: different holding periods and overlapping variants are not independent trades. Positive gross unit P&L means favorable invented marks under the frozen mechanics, not realized investor profit. The separately reported capacity haircut, capital, loss-risk and volume participation assumptions do not convert gross results into a net account return. Last-trade marks, standard contract multipliers and sparse stale-session carry do not establish executable spreads, fills or real capacity. No qualified asset exposure or drawdown series is available from these per-event rows.

In [ ]:
events = pd.read_csv(first_dir / "events.csv")
results = pd.read_csv(first_dir / "results.csv")
capacity = pd.read_csv(first_dir / "capacity.csv")
display(events)
display(results.head(12))
display(capacity)
display(Markdown("**Read these results:** one invented event supplies overlapping strategy/horizon rows. Gross unit P&L excludes account costs; capacity is a separate assumption table. These rows cannot supply a qualified win-rate, turnover or Sharpe denominator."))


In [ ]:
from src.config import STRATEGIES, STRATEGY_LABEL
expiry_rows = results.loc[(results["entry"] == "post")
                         & (results["horizon"].astype(str) == "exp")]
if expiry_rows.empty:
    print("No expiry comparison is available; no return is inferred.")
else:
    comparison = expiry_rows[STRATEGIES].iloc[0]
    fig, ax = plt.subplots(figsize=(10, 4))
    colors = ["#486581" if value >= 0 else "#b44b4b" for value in comparison]
    ax.bar([STRATEGY_LABEL[name] for name in comparison.index], comparison.values, color=colors)
    ax.axhline(0, color="#444444", linewidth=0.8)
    ax.set_ylabel("Gross unit P&L per dollar of entry spot")
    ax.set_title("Synthetic event only: post-entry comparison at expiry")
    ax.tick_params(axis="x", rotation=25)
    fig.tight_layout()
    fig.savefig(REPORT_DIR / "synthetic_strategy_mechanics.png", dpi=150)
    plt.show()
    plt.close(fig)

## 7. Canonical wording account workflow: fabricated engineering run

This separate section invokes the existing `scripts/run_simple_wording_backtest.py` CLI in **engineering** mode. It uses only the pinned **SYNTHETIC FABRICATED** candidate/quote fixture and the existing marked-account exporter. It adds no strategy or account engine.

The saved wording and baseline panels demonstrate the existing regular valuation grid and account mechanics. They are not observations of real trading performance. The existing controller preserves `canonical_economic_qualified=False` and `headline_eligible=False`, with P&L/Sharpe/win-rate/CI/edge headlines withheld. Qualified real inputs, market acceptance and complete approval evidence are not supplied by this public fixture.

CLI equivalent from the repository root (choose a fresh output directory):

    python scripts/run_simple_wording_backtest.py --root . --mode engineering --output-dir data/processed/wording-engineering/run-1


The frozen signal takes the wording sign: positive long, negative short, zero no wording trade. The matched baseline goes long on the same eligible opportunities, including the neutral opportunity; zero-yield idle cash is another stated reference. Integer shares target equal dollar exposure with a maximum gross fraction of one and no economic leverage. Short proceeds are restricted, borrow/collateral and fees require evidence, and an unseen exit quote must not change selection or entry sizing.

The fabricated fixture supplies invented quotes, entry/exit fees, borrow and margin values. These are engineering attestations, not accepted historical cost evidence. The schedule is public bound +60 seconds processing, first eligible exchange open +60 seconds entry, actual close -60 seconds exit. Timezones are UTC in the fixture; date-only options clocks cannot be substituted. The account is flat at regular close marks; that does not prove continuous intraday gross exposure, liquidity or collateral adequacy.

In each panel below, `nav` is synthetic marked account equity in USD and `net_return` is that account's period return. A no-trade row can carry idle cash; a missing row must stay missing. Read `reason` and `status` before interpreting values. Wording versus baseline differences arise from this fabricated sign/opportunity/cost wiring, not measured predictive skill. A smooth fixture equity path is not evidence of real low drawdown or capacity.

### Equity implementation and reuse across strategies and universes

The bundled canonical engineering route runs on **equities**: wording signal -> historical eligibility -> frozen long/short target policy -> executable quotes, costs and borrow -> single account ledger -> complete-calendar NAV and matched baseline -> qualified metrics. This fixture exercises the wiring with fabricated data; it does not qualify historical trading performance.

The architecture separates those responsibilities so the same account workflow can support other strategy definitions and security universes. Reuse requires each strategy to produce **causal trade intents** from accepted point-in-time features, and each universe to supply tested lifecycle adapters and a frozen protocol. It is not an arbitrary asset or strategy plug-in that becomes validated merely by producing a signal.

| Boundary | Current equity route | Required for another strategy or universe |
| --- | --- | --- |
| Signal and decision policy | Frozen wording sign, eligibility, timing, equal-dollar integer-share long/short targets | Accepted point-in-time features, past-only selection, causal intents and frozen sizing/entry/exit rules; test unavailable or late information |
| Security universe and market lifecycle | Dated equity identity, reference sessions, executable quotes and same-session corporate-action coverage | Dated security mapping, calendars/timezones, pricing units and multipliers, expiry/settlement/corporate-action lifecycle adapters and independent acceptance |
| Portfolio and execution policy | Existing single account ledger, cash, restricted short proceeds, borrow, margin/collateral and costs | Tested cash/margin/collateral and cost/financing lifecycle adapters; accepted fills/no-fills, capacity, exposure and account reconciliation |
| Comparison and reporting gates | Complete regular-calendar NAV, matched always-long/zero-cash baselines, withheld unqualified headlines | Same eligible opportunity set and valuation window, complete ledger/marks, relevant comparator and independently qualified metric evidence |

The separate options example reports gross unit mechanics and does not inherit equity account qualification. Futures or other universes need their own dated contract/unit/settlement and margin adapters, frozen strategy protocols and independent qualification. They should reuse the existing account engine through tested adapters rather than introduce a second account engine. A reusable interface preserves missing-evidence gates; it does not erase asset-specific risks or imply that every strategy is implemented here.


In [ ]:
import subprocess

wording_dir = REPORT_DIR / "wording-engineering"
wording_command = [sys.executable, str(ROOT / "scripts/run_simple_wording_backtest.py"),
                   "--root", str(ROOT), "--mode", "engineering", "--output-dir",
                   wording_dir.relative_to(ROOT).as_posix()]
wording_run = subprocess.run(wording_command, cwd=ROOT, check=True,
                             capture_output=True, text=True, timeout=120)
wording_report = json.loads((wording_dir / "report.json").read_text(encoding="utf-8"))
assert wording_report["mode"] == "engineering"
assert wording_report["status"] == "synthetic_contract_only"
assert wording_report["replay_status"] == "contract_replayed"
assert wording_report["assessment"]["eligible_count"] == 3
assert wording_report["canonical_economic_qualified"] is False
assert wording_report["headline_eligible"] is False
assert all(wording_report[name] is None for name in
           ("wording_pnl", "baseline_pnl", "wording_sharpe", "baseline_sharpe",
            "wording_win_rate", "baseline_win_rate", "confidence_interval", "edge_claim"))
display(pd.DataFrame([{name: wording_report[name] for name in
                      ("status", "mode", "label", "replay_status",
                       "canonical_economic_qualified", "headline_eligible")}]))
for arm in ("wording", "baseline"):
    saved = json.loads((wording_dir / f"panel_{arm}.json").read_text(encoding="utf-8"))
    assert saved == wording_report["panels"][arm]
    panel = saved["panel"]
    assert panel["scope"] == "synthetic_engineering"
    assert panel["status"] == "contract_exported"
    assert len(panel["rows"]) == 252
    print(f"{arm}: FABRICATED engineering account panel; status={panel['status']}")
    display(pd.DataFrame(panel["rows"]))
print("Canonical wording engineering CLI completed; qualified real performance remains missing.")

display(Markdown("**Read these account panels:** 252 close marks describe one fabricated opportunity set, not 252 independent events. Compare the same-opportunity baseline and check missing reasons. Fee/borrow/collateral attestations and flat-close exposure are engineering inputs; real turnover, intraday risk and drawdown remain unqualified."))


### What the requested performance metrics would mean

The options plot is gross per-dollar spot mechanics. The wording rows are fabricated marked-account mechanics. Neither establishes qualified real returns; do not calculate a headline metric from these demonstration tables.

| Metric | Definition and denominator | Current result and reason |
| --- | --- | --- |
| Net P&L | Full account final NAV minus initial NAV, adjusting external flows and including accepted fees, spreads, slippage, borrow and financing/collateral obligations | N/A: qualified complete market/cost/account evidence is absent; options unit P&L is gross and wording headlines are null |
| Sharpe | Mean daily excess account return divided by sample standard deviation, annualized by sqrt(252) for an accepted regular daily calendar/reference; undefined with fewer than two observations or zero volatility | N/A: fabricated returns are not qualified; frozen wording protocol additionally requires at least 30 periods and accepted evidence; no empirical CI supplied |
| Win rate | Profitable closed qualified trades / all closed qualified trades; use net-of-cost trade P&L, count break-even as non-winning and disclose exclusions | N/A: no closed qualified real-trade ledger; three fabricated candidates and overlapping options variants are not that denominator |
| Turnover | One-way gross traded notional (absolute purchases plus sales, counting each executed side once) / average valid account NAV over the same stated window; a half-round-trip convention would require a separate label | N/A: complete qualified execution ledger/NAV coverage is missing; do not infer trades from changes in sparse account marks |
| Maximum drawdown | Maximum (running peak NAV - current NAV) / running peak NAV on a complete positive, flow-adjusted account equity series | N/A: no qualified real NAV series; fabricated flat-close marks omit intraday risk and are not economic evidence |
| Exposure and sample size | Accepted position notional/NAV at named times; independent events and valuation periods reported separately | N/A real exposure: intraday evidence missing; one options event and three invented wording candidates only, despite 252 calendar rows |

The zero-cash reference, always-long baseline and options gross comparisons answer different questions. Any real metric would need the same calendar, costs, investable opportunities, universe/selection policy, publication/processing clocks and past-only information to prevent lookahead for both arms. Incomplete OCR and model coverage can bias selection toward available documents. A convenience subset, invented regime, overlapping horizons or hindsight classification cannot establish generalization, significance, regime robustness or alpha.

## 8. Outputs and next inputs needed

This run exports the complete storage catalog, data-area summary, team-reported coverage status table, historical table audit, package versions, two independent replay receipts/CSV sets, a synthetic mechanics figure, and the canonical wording engineering report/panels.

To reproduce an actual historical study, restore its exact permitted input bytes and validate their original hashes; restore the matching code/config/environment and owner-approved runner. A new live download is a new snapshot and may differ. Complete the source/clock/model/market/account qualification gates before interpreting economic results.

The PDF OCR/model workload remains incomplete under the team's reported compute/resource limitation. The notebook makes the available subset and missing evidence explicit. It does not run OCR, train models, open protected final-test labels, or infer results for unprocessed documents.

In [ ]:
summary = {
    "report_id": RUN_ID,
    "status": "completed_metadata_report_and_synthetic_replay",
    "catalog_files": len(catalog),
    "independent_replay_identical": first_hashes == second_hashes,
    "scientific_output_sha256": first_hashes,
    "economic_qualification": "not_established",
    "team_reported_ocr_limitation": "Insufficient available GPU compute and OCR/model processing resources to finish the full PDF workload; incomplete extracted and reviewed dataset.",
    "historical_status_counts": historical["status"].value_counts().to_dict(),
    "environment": runtime,
}
(REPORT_DIR / "notebook_summary.json").write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")
print("Report saved:", REPORT_DIR.relative_to(ROOT).as_posix())
print("Independent replay verified; actual corpus and financial qualification gaps remain explicit.")